In [79]:
import sys
import os
sys.path.append(os.path.abspath(".."))
import numpy as np
import pandas as pd
from data.market_data import get_multiple_tickers
from pricing.american import american_put_without_dividends, american_put_with_dividends, american_call_without_dividends, american_call_with_dividends
from pricing.european import heston_european_call_option, heston_european_put_option
from calibration.implied_vol import implied_volatility


##### tickers and market constraint

In [80]:
tickers = ['AAPL', 'NVDA']
r = 0.05
q=0
N= 10000
M = 100
spread_limit = 0.05

In [81]:
options_df = get_multiple_tickers(tickers, spread_limit=spread_limit)

pulling...AAPL...
pulling...NVDA...


In [82]:
options_df.head()

contractSymbol             lastTradeDate  strike  lastPrice    bid  \
0  AAPL260302C00190000 2026-03-02 19:49:28+00:00   190.0      75.83  73.40   
1  AAPL260302C00205000 2026-03-02 19:51:26+00:00   205.0      61.06  58.25   
2  AAPL260302C00250000 2026-03-02 20:27:16+00:00   250.0      14.35  13.90   
3  AAPL260302C00252500 2026-03-02 19:28:43+00:00   252.5      13.41  11.55   
4  AAPL260302C00255000 2026-03-02 20:10:20+00:00   255.0       9.80   8.85   

     ask    change  percentChange  volume  openInterest  impliedVolatility  \
0  75.95  2.580002       3.522187     3.0          29.0           4.074224   
1  60.95 -8.439999     -12.143883     7.0           3.0           3.299806   
2  14.05 -1.030000      -6.697007    23.0          22.0           0.000010   
3  11.75  0.559999       4.357972    41.0          25.0           0.000010   
4   9.00 -1.700000     -14.782607    74.0          64.0           0.000010   

   ticker ExerciseStyle mid_price         T rel_spread  moneyness  \
0    AAPL      american     2.245  0.005789   0.048998   0.998024   
1    AAPL      american     1.570  0.005789   0.038217   0.998024   
2    AAPL      american     5.075  0.011269   0.049261   0.987943   
3    AAPL      american     2.150  0.011269   0.046512   1.008105   
4    AAPL      american    18.550  0.016748   0.032345   0.927457   

  forward_spot disc_strike  lower_bound  
0   247.990005  247.428369     0.561636  
1   247.990005  247.428369     0.000000  
2   247.990005  244.861998     3.128008  
3   247.990005  249.859182     0.000000  
4   247.990005  229.807478    18.182528  

[5 rows x 26 columns]

#### Filtering for one option only to check for model flow

In [83]:
def price_row(row, r, q, heston_params ):
    S0 = row['spot']
    K=row['strike']
    T= row['T']
    option_type = row['type']

    v0, kappa, theta, sigma, rho = heston_params

    if option_type == 'call':
        if q==0:
            print('american_call_without_dividends ....exexcuted')
            return american_call_without_dividends(S0, K, T, r, v0, kappa, theta, sigma, rho)
        else:
            print('american_call_with_dividends....exexcuted')
            return american_call_with_dividends(S0, K, T, r, v0, kappa, theta, sigma, rho, M, N, q)
    elif option_type == 'put':
        if q==0:
            print('american_put_without_dividends....exexcuted')
            return american_put_without_dividends(S0, K, T, r, v0, kappa, theta, sigma, rho, M, N)
        else:
            print('american_put_with_dividends....exexcuted')
            return american_put_with_dividends(S0, K, T, r, v0, kappa, theta, sigma, rho, M, N, q)

In [84]:
calibration_df_nvda = options_df[options_df['ticker']=='NVDA'].copy()

calibration_df_nvda.head()

,contractSymbol,lastTradeDate,strike,lastPrice,bid,ask,change,percentChange,volume,openInterest,...,spot,ticker,ExerciseStyle,mid_price,T,rel_spread,moneyness,forward_spot,disc_strike,lower_bound
427,NVDA260323C00167500,2026-03-20 19:59:36+00:00,167.5,5.75,5.75,5.90,-5.90,-50.643772,295.0,36.0,...,172.699997,NVDA,american,5.825,0.005789,0.025751,0.969890,172.699997,167.451523,5.248474
428,NVDA260323C00170000,2026-03-20 19:59:57+00:00,170.0,3.70,3.75,3.80,-5.00,-57.471264,3976.0,238.0,...,172.699997,NVDA,american,3.775,0.005789,0.013245,0.984366,172.699997,169.950800,2.749197
429,NVDA260323C00172500,2026-03-20 19:59:53+00:00,172.5,2.06,2.06,2.08,-4.09,-66.504070,10011.0,591.0,...,172.699997,NVDA,american,2.070,0.005789,0.009662,0.998842,172.699997,172.450076,0.249921
430,NVDA260323C00175000,2026-03-20 20:00:00+00:00,175.0,0.90,0.89,0.91,-3.90,-81.250000,48364.0,4801.0,...,172.699997,NVDA,american,0.900,0.005789,0.022222,1.013318,172.699997,174.949353,0.000000
431,NVDA260323C00177500,2026-03-20 19:59:59+00:00,177.5,0.31,0.31,0.32,-2.58,-89.273360,47453.0,4761.0,...,172.699997,NVDA,american,0.315,0.005789,0.031746,1.027794,172.699997,177.448629,0.000000


In [85]:
calibration_df_nvda.shape

(699, 26)

In [13]:
initial_guess = [
    0.04,   # v0  (initial variance ≈ 20% vol)
    2.0,    # kappa
    0.04,   # theta
    0.5,    # sigma
    -0.7    # rho
]

In [15]:
options_df_nvda['heston_price'] = options_df_nvda.apply(lambda row: price_row(row, r, q, heston_params), axis=1)

american_call_without_dividends ....exexcuted


In [16]:
print("Calibrated Parameters:")
print("v0    =", params_opt[0])
print("kappa =", params_opt[1])
print("theta =", params_opt[2])
print("sigma =", params_opt[3])
print("rho   =", params_opt[4])
print("\nFinal Loss =", loss_val)

Calibrated Parameters:
v0    = 0.24403566968414625
kappa = 2.000169494947594
theta = 0.1087414325971798
sigma = 0.5085892157308337
rho   = -0.7084200062062825

Final Loss = 0.03448939252321163


In [86]:
calibrated_heston_params = (0.24403566968414625, 2.000169494947594, 0.1087414325971798, 0.5085892157308337, -0.7084200062062825)

In [87]:
options_df_nvda = options_df[options_df['ticker']=='NVDA'].copy()
options_df_nvda['calibrated_heston_price'] = options_df_nvda.apply(lambda row: price_row(row, r, q, calibrated_heston_params), axis=1)
options_df_nvda.head()

/Users/subhamkhinchi/Downloads/Downloads - NCSU/Projects/Heston Engine/simulation/lsmc.py:25: RankWarning: Polyfit may be poorly conditioned
  coeffs = np.polyfit(X[itm], Y[itm], 2)
/Users/subhamkhinchi/Downloads/Downloads - NCSU/Projects/Heston Engine/simulation/lsmc.py:25: RankWarning: Polyfit may be poorly conditioned
  coeffs = np.polyfit(X[itm], Y[itm], 2)
/Users/subhamkhinchi/Downloads/Downloads - NCSU/Projects/Heston Engine/simulation/lsmc.py:25: RankWarning: Polyfit may be poorly conditioned
  coeffs = np.polyfit(X[itm], Y[itm], 2)
/Users/subhamkhinchi/Downloads/Downloads - NCSU/Projects/Heston Engine/simulation/lsmc.py:25: RankWarning: Polyfit may be poorly conditioned
  coeffs = np.polyfit(X[itm], Y[itm], 2)
/Users/subhamkhinchi/Downloads/Downloads - NCSU/Projects/Heston Engine/simulation/lsmc.py:25: RankWarning: Polyfit may be poorly conditioned
  coeffs = np.polyfit(X[itm], Y[itm], 2)
/Users/subhamkhinchi/Downloads/Downloads - NCSU/Projects/Heston Engine/simulation/lsmc.py:

,contractSymbol,lastTradeDate,strike,lastPrice,bid,ask,change,percentChange,volume,openInterest,...,ticker,ExerciseStyle,mid_price,T,rel_spread,moneyness,forward_spot,disc_strike,lower_bound,calibrated_heston_price
427,NVDA260323C00167500,2026-03-20 19:59:36+00:00,167.5,5.75,5.75,5.90,-5.90,-50.643772,295.0,36.0,...,NVDA,american,5.825,0.005789,0.025751,0.969890,172.699997,167.451523,5.248474,6.005878
428,NVDA260323C00170000,2026-03-20 19:59:57+00:00,170.0,3.70,3.75,3.80,-5.00,-57.471264,3976.0,238.0,...,NVDA,american,3.775,0.005789,0.013245,0.984366,172.699997,169.950800,2.749197,4.183347
429,NVDA260323C00172500,2026-03-20 19:59:53+00:00,172.5,2.06,2.06,2.08,-4.09,-66.504070,10011.0,591.0,...,NVDA,american,2.070,0.005789,0.009662,0.998842,172.699997,172.450076,0.249921,2.710138
430,NVDA260323C00175000,2026-03-20 20:00:00+00:00,175.0,0.90,0.89,0.91,-3.90,-81.250000,48364.0,4801.0,...,NVDA,american,0.900,0.005789,0.022222,1.013318,172.699997,174.949353,0.000000,1.617317
431,NVDA260323C00177500,2026-03-20 19:59:59+00:00,177.5,0.31,0.31,0.32,-2.58,-89.273360,47453.0,4761.0,...,NVDA,american,0.315,0.005789,0.031746,1.027794,172.699997,177.448629,0.000000,0.881776


In [88]:
options_df_nvda_vol = options_df_nvda.copy()
options_df_nvda_vol.head()

,contractSymbol,lastTradeDate,strike,lastPrice,bid,ask,change,percentChange,volume,openInterest,...,ticker,ExerciseStyle,mid_price,T,rel_spread,moneyness,forward_spot,disc_strike,lower_bound,calibrated_heston_price
427,NVDA260323C00167500,2026-03-20 19:59:36+00:00,167.5,5.75,5.75,5.90,-5.90,-50.643772,295.0,36.0,...,NVDA,american,5.825,0.005789,0.025751,0.969890,172.699997,167.451523,5.248474,6.005878
428,NVDA260323C00170000,2026-03-20 19:59:57+00:00,170.0,3.70,3.75,3.80,-5.00,-57.471264,3976.0,238.0,...,NVDA,american,3.775,0.005789,0.013245,0.984366,172.699997,169.950800,2.749197,4.183347
429,NVDA260323C00172500,2026-03-20 19:59:53+00:00,172.5,2.06,2.06,2.08,-4.09,-66.504070,10011.0,591.0,...,NVDA,american,2.070,0.005789,0.009662,0.998842,172.699997,172.450076,0.249921,2.710138
430,NVDA260323C00175000,2026-03-20 20:00:00+00:00,175.0,0.90,0.89,0.91,-3.90,-81.250000,48364.0,4801.0,...,NVDA,american,0.900,0.005789,0.022222,1.013318,172.699997,174.949353,0.000000,1.617317
431,NVDA260323C00177500,2026-03-20 19:59:59+00:00,177.5,0.31,0.31,0.32,-2.58,-89.273360,47453.0,4761.0,...,NVDA,american,0.315,0.005789,0.031746,1.027794,172.699997,177.448629,0.000000,0.881776


#### Market IV

In [89]:
options_df_nvda_vol["market_iv"] = options_df_nvda_vol.apply(
    lambda row: implied_volatility(
        row["mid_price"],
        row["spot"],
        row["strike"],
        r,
        row["T"],
        row["type"],
        q
    ),
    axis=1
)
options_df_nvda_vol.head()

,contractSymbol,lastTradeDate,strike,lastPrice,bid,ask,change,percentChange,volume,openInterest,...,ExerciseStyle,mid_price,T,rel_spread,moneyness,forward_spot,disc_strike,lower_bound,calibrated_heston_price,market_iv
427,NVDA260323C00167500,2026-03-20 19:59:36+00:00,167.5,5.75,5.75,5.90,-5.90,-50.643772,295.0,36.0,...,american,5.825,0.005789,0.025751,0.969890,172.699997,167.451523,5.248474,6.005878,0.448046
428,NVDA260323C00170000,2026-03-20 19:59:57+00:00,170.0,3.70,3.75,3.80,-5.00,-57.471264,3976.0,238.0,...,american,3.775,0.005789,0.013245,0.984366,172.699997,169.950800,2.749197,4.183347,0.408313
429,NVDA260323C00172500,2026-03-20 19:59:53+00:00,172.5,2.06,2.06,2.08,-4.09,-66.504070,10011.0,591.0,...,american,2.070,0.005789,0.009662,0.998842,172.699997,172.450076,0.249921,2.710138,0.370832
430,NVDA260323C00175000,2026-03-20 20:00:00+00:00,175.0,0.90,0.89,0.91,-3.90,-81.250000,48364.0,4801.0,...,american,0.900,0.005789,0.022222,1.013318,172.699997,174.949353,0.000000,1.617317,0.342347
431,NVDA260323C00177500,2026-03-20 19:59:59+00:00,177.5,0.31,0.31,0.32,-2.58,-89.273360,47453.0,4761.0,...,american,0.315,0.005789,0.031746,1.027794,172.699997,177.448629,0.000000,0.881776,0.330499


In [90]:
# run diagnostics

def diagnose_iv_row(row, price_col='mid_price'):
    try:
        S = float(row['spot']); K = float(row['strike']); T = float(row['T']); price = float(row[price_col])
    except Exception:
        return 'bad_input_types'
    if price <= 0:
        return 'price<=0'
    if T <= 0:
        return 'T<=0'
    forward_spot = S * np.exp(-q * T)
    disc_strike = K * np.exp(-r * T)
    if row['type'].lower() == 'call':
        lower = max(0.0, forward_spot - disc_strike); upper = forward_spot
    else:
        lower = max(0.0, disc_strike - forward_spot); upper = disc_strike
    tol = 1e-12
    if (price + tol) < lower or (price - tol) > upper:
        return ('outside_arbitrage_bounds', price, lower, upper)
    iv = implied_volatility(price, S, K, r, T, row['type'], q)
    if np.isnan(iv):
        return 'bracketing_failed_or_other'
    return 'ok'

options_df_nvda_vol['iv_diag'] = options_df_nvda_vol.apply(diagnose_iv_row, axis=1)

In [91]:
options_df_nvda_vol['iv_diag'].value_counts(dropna=False)

iv_diag
ok    699
Name: count, dtype: int64

#### Calibrated Model IV

In [92]:
options_df_nvda_vol["model_iv"] = options_df_nvda_vol.apply(
    lambda row: implied_volatility(
        row["heston_price"],
        row["spot"],
        row["strike"],
        row["T"],
        r,
        row["type"],
        q
    ),
    axis=1
)
options_df_nvda_vol.head()

,contractSymbol,lastTradeDate,strike,lastPrice,bid,ask,change,percentChange,volume,openInterest,...,T,rel_spread,moneyness,forward_spot,disc_strike,lower_bound,calibrated_heston_price,market_iv,iv_diag,model_iv
427,NVDA260323C00167500,2026-03-20 19:59:36+00:00,167.5,5.75,5.75,5.90,-5.90,-50.643772,295.0,36.0,...,0.005789,0.025751,0.969890,172.699997,167.451523,5.248474,6.005878,0.448046,ok,0.498679
428,NVDA260323C00170000,2026-03-20 19:59:57+00:00,170.0,3.70,3.75,3.80,-5.00,-57.471264,3976.0,238.0,...,0.005789,0.013245,0.984366,172.699997,169.950800,2.749197,4.183347,0.408313,ok,0.495960
429,NVDA260323C00172500,2026-03-20 19:59:53+00:00,172.5,2.06,2.06,2.08,-4.09,-66.504070,10011.0,591.0,...,0.005789,0.009662,0.998842,172.699997,172.450076,0.249921,2.710138,0.370832,ok,0.493172
430,NVDA260323C00175000,2026-03-20 20:00:00+00:00,175.0,0.90,0.89,0.91,-3.90,-81.250000,48364.0,4801.0,...,0.005789,0.022222,1.013318,172.699997,174.949353,0.000000,1.617317,0.342347,ok,0.490524
431,NVDA260323C00177500,2026-03-20 19:59:59+00:00,177.5,0.31,0.31,0.32,-2.58,-89.273360,47453.0,4761.0,...,0.005789,0.031746,1.027794,172.699997,177.448629,0.000000,0.881776,0.330499,ok,0.488063


In [98]:
options_df_nvda_vol.columns

Index(['contractSymbol', 'lastTradeDate', 'strike', 'lastPrice', 'bid', 'ask',
       'change', 'percentChange', 'volume', 'openInterest',
       'impliedVolatility', 'inTheMoney', 'contractSize', 'currency', 'type',
       'maturity', 'spot', 'ticker', 'ExerciseStyle', 'mid_price', 'T',
       'rel_spread', 'moneyness', 'forward_spot', 'disc_strike', 'lower_bound',
       'calibrated_heston_price', 'market_iv', 'iv_diag', 'model_iv'],
      dtype='object')

In [93]:
from scipy.interpolate import griddata

M = options_df_nvda_vol["moneyness"].values
T = options_df_nvda_vol["T"].values
IV = options_df_nvda_vol["market_iv"].values

# grid
m_grid = np.linspace(M.min(), M.max(), 50)
t_grid = np.linspace(T.min(), T.max(), 50)

M_grid, T_grid = np.meshgrid(m_grid, t_grid)

IV_grid = griddata(
    (M, T),
    IV,
    (M_grid, T_grid),
    method="cubic"
)

In [94]:
import plotly.graph_objects as go

fig = go.Figure(
    data=[
        go.Surface(
            x=M_grid,
            y=T_grid,
            z=IV_grid,
            colorscale="Jet"
        )
    ]
)

fig.update_layout(
    title="Implied Volatility Surface",
    scene=dict(
        xaxis_title="Moneyness (S/K)",
        yaxis_title="Time to Maturity",
        zaxis_title="Implied Volatility"
    )
)

fig.show()